In [ ]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["ScfRestrictedDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

VeloxChem 1.0rc4
Environment OK


In [ ]:
import re

def read_pdb_coordinates(pdb_path, chain=None, residue_name=None):
    """Parse a PDB file and return atom symbols + coordinates."""
    atoms = []
    coords = []

    with open(pdb_path, "r") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                rec_chain = line[21].strip()
                rec_resname = line[17:20].strip()

                if chain is not None and rec_chain != chain:
                    continue
                if residue_name is not None and rec_resname != residue_name:
                    continue

                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                element = line[76:78].strip()
                if not element:
                    atom_name = line[12:16].strip()
                    element = "".join([c for c in atom_name if c.isalpha()])[:2]
                    element = element.capitalize()

                atoms.append(element)
                coords.append((x, y, z))

    return atoms, np.array(coords)


def read_formal_charge(pdb_path, default=0):
    """Pull 'REMARK  15 TOTAL FORMAL CHARGE +N' from the PDB header, if present."""
    pattern = re.compile(r"TOTAL FORMAL CHARGE\s*([+-]?\d+)")
    with open(pdb_path, "r") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                return int(match.group(1))
    return default


def to_xyz_string(atoms, coords, comment="Structure from PDB"):
    lines = [str(len(atoms)), comment]
    for atom, (x, y, z) in zip(atoms, coords):
        lines.append(f"{atom:2s} {x:12.6f} {y:12.6f} {z:12.6f}")
    return "\n".join(lines)


# --- Files ---
pdb_files = {
    "RS": "step_1_1_RS.pdb",
    "TS": "step_1_1_TS.pdb",
    "PS": "step_1_1_PS.pdb",
}

# These PDBs already contain only the QM region (substrate + capped amine
# + catalytic waters), so no residue filtering is needed here.
residue_filter = None

# Assumed spin state for all structures (closed-shell singlet).
# Change per-structure below if that's not the case.
multiplicity = 1

molecules = {}
charges = {}

for label, path in pdb_files.items():
    atoms, coords = read_pdb_coordinates(path, residue_name=residue_filter)
    xyz_string = to_xyz_string(atoms, coords, comment=f"{label} structure")

    mol = vlx.Molecule.read_xyz_string(xyz_string)

    charge = read_formal_charge(path, default=0)
    mol.set_charge(charge)
    mol.set_multiplicity(multiplicity)

    molecules[label] = mol
    charges[label] = charge

    print(f"{label}: {len(atoms)} atoms, charge={charge:+d}, multiplicity={multiplicity} ({path})")

# Access each one:
molecule_rs = molecules["RS"]
molecule_ts = molecules["TS"]
molecule_ps = molecules["PS"]

print()
print(molecule_rs.get_xyz_string())
molecule_rs.show(atom_indices=True, width=520, height=360)


RS: 57 atoms, charge=+0, multiplicity=1 (step_1_1_RS.pdb)
TS: 57 atoms, charge=+0, multiplicity=1 (step_1_1_TS.pdb)
PS: 57 atoms, charge=+0, multiplicity=1 (step_1_1_PS.pdb)

57

N              0.000000000000         0.000000000000         0.000000000000
C              1.460000000000         0.000000000000         0.000000000000
C              2.155000000000         0.774000000000         0.922000000000
C              3.545000000000         0.774000000000         0.922000000000
C              4.240000000000        -0.000000000000        -0.000000000000
C              3.545000000000        -0.774000000000        -0.922000000000
C              2.155000000000        -0.774000000000        -0.922000000000
C              5.740000000000        -0.000000000000        -0.000000000000
C              6.240000000000         1.083000000000        -0.909000000000
C              0.967000000000         0.010000000000        -2.840000000000
C              1.720000000000         1.308000000000        -

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# ============================================================
# run_mode = "quick" -> fast sanity check: single-point HF/def2-svp SCF
#                        on the raw PDB geometry, no optimization, no
#                        dispersion, no Hessian. Use this first to confirm
#                        charges/multiplicity/basis are all set up correctly.
# run_mode = "full"  -> the real calculation: B3LYP-D geometry optimization
#                        followed by a converged SCF at the optimized
#                        geometry (feeds the RESP-charges, vibrational-
#                        analysis, and barrier cells below).
#
# For "full", RS and PS are minimized normally. TS uses a genuine
# transition-state search (partitioned rational-function optimization),
# which requires Hessian information to follow the negative-curvature
# mode -- this is NOT the same as an unconstrained minimization and is
# why "transition" + "hessian" must both be set for TS.
# ============================================================
run_mode = "full"   # change to "full" once the quick check looks good

basis_label = "def2-svp"

opt_molecules = {}
opt_results_all = {}
scf_results_all = {}
energies_all = {}   # electronic energy (Hartree) per label

for label, mol in molecules.items():
    print(f"=== {run_mode.upper()} run: {label} (charge={charges[label]:+d}) ===")

    basis = vlx.MolecularBasis.read(mol, basis_label)
    scf_drv = vlx.ScfRestrictedDriver()
    scf_drv.ostream.mute()

    if run_mode == "quick":
        # Default driver settings = plain HF, no dispersion, no optimization.
        scf_results = scf_drv.compute(mol, basis)

        opt_molecules[label] = mol          # geometry unchanged (raw PDB coords)
        scf_results_all[label] = scf_results
        opt_results_all[label] = None
        energies_all[label] = scf_drv.get_scf_energy()

        print(f"HF/{basis_label} energy ({label}): {energies_all[label]: .8f} Hartree")

    elif run_mode == "full":
        scf_drv.xcfun = "b3lyp"
        scf_drv.dispersion = True

        opt_drv = vlx.OptimizationDriver(scf_drv)
        opt_drv.ostream.mute()
        opt_drv.max_iter = 100

        if label == "TS":
            # Genuine saddle-point search, not a minimization.
            if hasattr(opt_drv, "transition"):
                opt_drv.transition = True
                opt_drv.hessian = "first+last"  # Hessian needed to follow the TS mode
            else:
                print("WARNING: this VeloxChem version does not expose "
                      "OptimizationDriver.transition -- falling back to a "
                      "plain minimization, which will NOT give a valid TS.")
        else:
            opt_drv.transition = False if hasattr(opt_drv, "transition") else None

        opt_results = opt_drv.compute(mol, basis)
        opt_molecule = opt_results["final_molecule"]

        # Converged SCF at the optimized geometry, needed for RESP fitting
        # and to get the final electronic energy at the stationary point.
        scf_results = scf_drv.compute(opt_molecule, basis)

        opt_molecules[label] = opt_molecule
        opt_results_all[label] = opt_results
        scf_results_all[label] = scf_results
        energies_all[label] = scf_drv.get_scf_energy()

        print(f"Optimized geometry ({label}):")
        print(opt_molecule.get_xyz_string())
        print("Optimization steps:", len(opt_results.get("opt_energies", [])))
        print(f"B3LYP-D/{basis_label} energy ({label}): {energies_all[label]: .8f} Hartree")

    else:
        raise ValueError(f"Unknown run_mode: {run_mode!r}. Use 'quick' or 'full'.")

    print()

opt_molecules["RS"].show(atom_indices=True, width=520, height=360)


=== QUICK run: RS (charge=+0) ===
HF/def2-svp energy (RS): -1236.32230998 Hartree

=== QUICK run: TS (charge=+0) ===
HF/def2-svp energy (TS): -1236.25387092 Hartree

=== QUICK run: PS (charge=+0) ===
HF/def2-svp energy (PS): -1236.24521600 Hartree



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
HARTREE_TO_KCAL = 627.5094740631
HARTREE_TO_KJ = 2625.4996394799

e_rs = energies_all["RS"]
e_ts = energies_all["TS"]
e_ps = energies_all["PS"]

forward_barrier = e_ts - e_rs
reverse_barrier = e_ts - e_ps
reaction_energy = e_ps - e_rs

print("Electronic energies (Hartree):")
print(f"  RS: {e_rs: .8f}")
print(f"  TS: {e_ts: .8f}")
print(f"  PS: {e_ps: .8f}")
print()
print(f"Forward barrier   (TS - RS): {forward_barrier * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({forward_barrier * HARTREE_TO_KJ:8.2f} kJ/mol)")
print(f"Reverse barrier   (TS - PS): {reverse_barrier * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({reverse_barrier * HARTREE_TO_KJ:8.2f} kJ/mol)")
print(f"Reaction energy   (PS - RS): {reaction_energy * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({reaction_energy * HARTREE_TO_KJ:8.2f} kJ/mol)")

if run_mode == "quick":
    print()
    print("Note: these are HF single-point energies on the RAW (unoptimized) PDB "
          "geometries -- useful only as a sanity check, not a real barrier.")


Electronic energies (Hartree):
  RS: -1236.32230998
  TS: -1236.25387092
  PS: -1236.24521600

Forward barrier   (TS - RS):    42.95 kcal/mol  (  179.69 kJ/mol)
Reverse barrier   (TS - PS):    -5.43 kcal/mol  (  -22.72 kJ/mol)
Reaction energy   (PS - RS):    48.38 kcal/mol  (  202.41 kJ/mol)

Note: these are HF single-point energies on the RAW (unoptimized) PDB geometries -- useful only as a sanity check, not a real barrier.


In [ ]:
resp_charges_all = {}

for label, opt_molecule in opt_molecules.items():
    print(f"=== RESP charges: {label} ===")

    basis = vlx.MolecularBasis.read(opt_molecule, basis_label)
    scf_results = scf_results_all[label]

    resp_drv = vlx.RespChargesDriver()
    resp_drv.ostream.mute()

    resp_charges = resp_drv.compute(opt_molecule, basis, scf_results, "resp")
    resp_charges_all[label] = resp_charges

    labels = list(opt_molecule.get_labels())
    print(f"{'Atom':>6} {'Charge':>10}")
    for atom_label, q in zip(labels, resp_charges):
        print(f"{atom_label:>6} {q:>10.4f}")
    print(f"Sum of charges: {sum(resp_charges):.4f}  (should equal net charge {charges[label]:+d})")
    print()


=== RESP charges: RS ===
  Atom     Charge
     N    -0.7182
     C     0.1372
     C    -0.1754
     C    -0.2180
     C     0.0728
     C    -0.2363
     C    -0.1196
     C    -0.1861
     C     0.0168
     C     0.3138
     C    -0.2940
     C    -0.1581
     C    -0.0246
     C     0.1014
     C    -0.0727
     O    -0.4403
     O    -0.8386
     H     0.3503
     H     0.3245
     H     0.1381
     H     0.1432
     H     0.2051
     H     0.1469
     H     0.1075
     H     0.1075
     H     0.0756
     H     0.2769
     H     0.2486
     H     0.0992
     H     0.2644
     H     0.1527
     H     0.0307
     H     0.0307
     H    -0.0158
     H    -0.0158
     H     0.0108
     H     0.0108
     H     0.0108
     H     0.4233
     H     0.3950
     O    -0.7889
     H     0.3863
     H     0.3852
     N    -0.4755
     C     0.5146
     O    -0.5112
     C    -0.0125
     C     0.5687
     O    -0.5404
     H     0.0107
     H     0.0107
     H     0.0107
     N    -0.3527
   

In [ ]:
vib_results_all = {}

for label, opt_molecule in opt_molecules.items():
    print(f"=== Vibrational analysis: {label} ===")

    basis = vlx.MolecularBasis.read(opt_molecule, basis_label)

    scf_drv = vlx.ScfRestrictedDriver()
    scf_drv.xcfun = "b3lyp"
    scf_drv.dispersion = True
    scf_drv.ostream.mute()
    scf_drv.compute(opt_molecule, basis)

    vib_drv = vlx.VibrationalAnalysis(scf_drv)
    vib_drv.ostream.mute()
    vib_drv.do_ir = True

    vib_results = vib_drv.compute(opt_molecule, basis)
    vib_results_all[label] = vib_results

    frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
    reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
    force_constants = np.array(vib_results["force_constants"], dtype=float)
    ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

    print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
    for i, freq in enumerate(frequencies, start=1):
        print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

    if label == "TS":
        n_imaginary = int(np.sum(frequencies < 0))
        print(f"\nImaginary frequencies for TS: {n_imaginary} (expect exactly 1 for a valid transition state)")
    print()


=== Vibrational analysis: RS ===


In [ ]:
CM1_TO_HARTREE = 4.5563352529120e-6  # 1 cm^-1 in Hartree

zpe_all = {}
for label, vib_results in vib_results_all.items():
    freqs = np.array(vib_results["vib_frequencies"], dtype=float)
    real_modes = freqs[freqs > 0]  # exclude the imaginary mode(s), e.g. TS's reaction coordinate
    zpe_all[label] = 0.5 * np.sum(real_modes) * CM1_TO_HARTREE
    n_imag = int(np.sum(freqs < 0))
    print(f"{label}: ZPE = {zpe_all[label]:.6f} Hartree ({zpe_all[label] * HARTREE_TO_KCAL:.2f} kcal/mol), "
          f"{n_imag} imaginary mode(s)")

print()

e_rs_zpe = energies_all["RS"] + zpe_all["RS"]
e_ts_zpe = energies_all["TS"] + zpe_all["TS"]
e_ps_zpe = energies_all["PS"] + zpe_all["PS"]

forward_barrier_zpe = e_ts_zpe - e_rs_zpe
reverse_barrier_zpe = e_ts_zpe - e_ps_zpe
reaction_energy_zpe = e_ps_zpe - e_rs_zpe

print("ZPE-corrected (electronic + ZPE):")
print(f"Forward barrier   (TS - RS): {forward_barrier_zpe * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({forward_barrier_zpe * HARTREE_TO_KJ:8.2f} kJ/mol)")
print(f"Reverse barrier   (TS - PS): {reverse_barrier_zpe * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({reverse_barrier_zpe * HARTREE_TO_KJ:8.2f} kJ/mol)")
print(f"Reaction energy   (PS - RS): {reaction_energy_zpe * HARTREE_TO_KCAL:8.2f} kcal/mol  "
      f"({reaction_energy_zpe * HARTREE_TO_KJ:8.2f} kJ/mol)")

if zpe_all.get("TS", 0) and n_imag != 1:
    print()
    print("Warning: a valid transition state should have exactly ONE imaginary "
          "frequency. Check the TS optimization if this isn't the case.")


In [ ]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, mode, amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Choose which structure and which mode to visualize
struct_label = "RS"     # "RS", "TS", or "PS"
mode_index = 0           # Python index: 0 is the first vibrational mode

vib_results = vib_results_all[struct_label]
frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)

print(f"Showing {struct_label} mode {mode_index}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecules[struct_label], normal_modes[mode_index], amplitude=0.45, frames=32).show()
print(normal_modes[mode_index])
